# ⚙️ Notebook - Prétraitement YOLOv8 avec décompression des masques

Cette version corrige la clé utilisée (`objects`), ajoute la décompression `zlib`,
et génère les fichiers `.txt` au format YOLOv8 (segments).

In [ ]:
def preprocessing(
    input_images_dir: str,
    input_masks_dir: str,
    output_images_dir: str,
    output_labels_dir: str,
    output_size: tuple = (640, 640)
):
    import os, json, base64, zlib, io
    from pathlib import Path
    from PIL import Image
    import numpy as np
    import cv2

    def decode_bitmap_to_mask_zlib(data_base64, origin, image_shape):
        """Décode un masque bitmap zlib+base64 et le place dans un canevas à la bonne position."""
        mask_bytes = base64.b64decode(data_base64)
        decompressed = zlib.decompress(mask_bytes)
        mask_img = Image.open(io.BytesIO(decompressed)).convert("L")
        mask_arr = np.array(mask_img)

        h, w = image_shape
        full_mask = np.zeros((h, w), dtype=np.uint8)

        x0, y0 = origin  # ⚠️ origin = (x, y)
        mh, mw = mask_arr.shape
        x1 = min(x0 + mw, w)
        y1 = min(y0 + mh, h)

        # ⚠️ Vérification pour éviter erreur de broadcast
        if x1 <= x0 or y1 <= y0:
            print(f"[⚠️] Masque hors cadre (origin: {origin}, shape: {mask_arr.shape}) — ignoré.")
            return full_mask

        mask_crop = mask_arr[:y1 - y0, :x1 - x0]
        full_mask[y0:y1, x0:x1] = mask_crop
        return full_mask

    def extract_yolo_vectors_from_mask(mask, class_id):
        """Extrait les contours du masque au format YOLOv8 segments."""
        h, w = mask.shape
        _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        annotations = []
        for contour in contours:
            contour = contour.squeeze()
            if len(contour.shape) != 2 or len(contour) < 3:
                continue
            norm_pts = [(x / w, y / h) for x, y in contour]
            flat = [str(round(coord, 6)) for pt in norm_pts for coord in pt]
            line = f"{class_id} " + " ".join(flat)
            annotations.append(line)
        return annotations

    # 📁 Initialisation des chemins
    input_images_dir = Path(input_images_dir)
    input_masks_dir = Path(input_masks_dir)
    output_images_dir = Path(output_images_dir)
    output_labels_dir = Path(output_labels_dir)
    output_images_dir.mkdir(parents=True, exist_ok=True)
    output_labels_dir.mkdir(parents=True, exist_ok=True)

    # 🔢 Mapping classTitle → class_id
    class_map = {}
    next_class_id = 0

    for json_file in input_masks_dir.glob("*.json"):
        with open(json_file, "r") as f:
            data = json.load(f)

        image_basename = json_file.name.replace(".jpg.json", "")
        image_filename = input_images_dir / f"{image_basename}.jpg"

        if not image_filename.exists():
            print(f"[❌] Image manquante pour {json_file.name}")
            continue

        # 📥 Redimensionnement de l'image
        image = Image.open(image_filename).convert("RGB")
        image = image.resize(output_size)
        image_out_path = output_images_dir / image_filename.name
        image.save(image_out_path)

        mask_shape = output_size[::-1]  # (height, width)
        all_vectors = []

        # 🔁 Parcourir les objets dans le JSON
        for obj in data.get("objects", []):
            bitmap = obj.get("bitmap")
            class_label = obj.get("classTitle", "unknown")

            # Attribuer un ID unique à chaque classe
            if class_label not in class_map:
                class_map[class_label] = next_class_id
                next_class_id += 1
            class_id = class_map[class_label]

            if bitmap:
                mask = decode_bitmap_to_mask_zlib(bitmap["data"], bitmap["origin"], mask_shape)
                vectors = extract_yolo_vectors_from_mask(mask, class_id)
                all_vectors.extend(vectors)

        # ✍️ Sauvegarde des vecteurs dans un .txt
        label_path = output_labels_dir / (json_file.stem + ".txt")
        with open(label_path, "w") as f:
            for line in all_vectors:
                f.write(line + "/n")

        print(f"✅ {json_file.name} → {len(all_vectors)} segments — classes : {set(class_map.values())}")


In [ ]:
# VAL
preprocessing(
    input_images_dir="C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/preprocessing/val/input/images",
    input_masks_dir="C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/preprocessing/val/input/ann",
    output_images_dir="C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/preprocessing/val/output/images_out",
    output_labels_dir="C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/preprocessing/val/output/labels_out",
    output_size=(640, 640)
)

✅ 00000048.jpg.json → 5 segments — classes : {0, 1, 2, 3, 4}
✅ 00000263.jpg.json → 3 segments — classes : {0, 1, 2, 3, 4, 5}
✅ 00001977.jpg.json → 5 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
✅ 00002106.jpg.json → 6 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13}
[⚠️] Masque hors cadre (origin: [690, 433], shape: (193, 292)) — ignoré.
✅ 00004401.jpg.json → 2 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14}
✅ 00004402.jpg.json → 2 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15}
✅ 00004403.jpg.json → 6 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19}
✅ 00004404.jpg.json → 5 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22}
✅ 00004405.jpg.json → 3 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22}
✅ 00004406.jpg.json → 13 segments — classes : {0, 1, 2, 3, 4, 

In [3]:
# TEST
preprocessing(
    input_images_dir="C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/preprocessing/test/input/img",
    input_masks_dir="C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/preprocessing/test/input/ann",
    output_images_dir="C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/preprocessing/test/output/images_out",
    output_labels_dir="C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/preprocessing/test/output/labels_txt",
    output_size=(640, 640)
)

✅ 00005586.jpg.json → 3 segments — classes : {0, 1, 2}
✅ 00005587.jpg.json → 3 segments — classes : {0, 1, 2, 3, 4}
✅ 00005588.jpg.json → 6 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8}
✅ 00005589.jpg.json → 2 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
✅ 00005590.jpg.json → 4 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12}
✅ 00005591.jpg.json → 10 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17}
✅ 00005592.jpg.json → 3 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18}
✅ 00005593.jpg.json → 10 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19}
[⚠️] Masque hors cadre (origin: [670, 106], shape: (918, 823)) — ignoré.
[⚠️] Masque hors cadre (origin: [704, 73], shape: (820, 691)) — ignoré.
✅ 00005594.jpg.json → 2 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21}
✅ 00005595.jpg.json → 5 segments — c

In [4]:
# TRAIN
preprocessing(
    input_images_dir="C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/preprocessing/train/input/img",
    input_masks_dir="C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/preprocessing/train/input/ann",
    output_images_dir="C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/preprocessing/train/output/images_out",
    output_labels_dir="C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/preprocessing/train/output/labels_out",
    output_size=(640, 640)
)

✅ 00000000.jpg.json → 3 segments — classes : {0, 1, 2}
✅ 00000001.jpg.json → 5 segments — classes : {0, 1, 2, 3, 4, 5, 6}
✅ 00000002.jpg.json → 2 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8}
✅ 00000003.jpg.json → 5 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11}
✅ 00000004.jpg.json → 6 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14}
✅ 00000005.jpg.json → 5 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16}
✅ 00000006.jpg.json → 7 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18}
[⚠️] Masque hors cadre (origin: [640, 452], shape: (1170, 826)) — ignoré.
[⚠️] Masque hors cadre (origin: [84, 1002], shape: (862, 837)) — ignoré.
✅ 00000007.jpg.json → 1 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18}
✅ 00000008.jpg.json → 7 segments — classes : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21}
✅ 00000009.jpg.json → 3 se

In [5]:
!pip install ultralytics

In [6]:
!pip install opencv-python torch torchvision albumentations

In [12]:
import os

# Dossier contenant les annotations
labels_dir = "C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/test"  # Modifie avec le bon chemin

# Parcours tous les fichiers du dossier
for filename in os.listdir(labels_dir):
    if filename.endswith(".jpg.txt"):  # Vérifie si le fichier a la mauvaise extension
        old_path = os.path.join(labels_dir, filename)
        new_filename = filename.replace(".jpg.txt", ".txt")
        new_path = os.path.join(labels_dir, new_filename)
        
        os.rename(old_path, new_path)  # Renomme le fichier
        print(f"Renommé : {old_path} → {new_path}")

print("Renommage terminé ✅")

Renommé : C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/test\00005586.jpg.txt → C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/test\00005586.txt
Renommé : C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/test\00005587.jpg.txt → C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/test\00005587.txt
Renommé : C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/test\00005588.jpg.txt → C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/test\00005588.txt
Renommé : C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/test\00005589.jpg.txt → C:/Users/fanch/OneDrive/Bu

In [13]:
# Dossier contenant les annotations TRAIN
labels_dir = "C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/train"  # Modifie avec le bon chemin

# Parcours tous les fichiers du dossier
for filename in os.listdir(labels_dir):
    if filename.endswith(".jpg.txt"):  # Vérifie si le fichier a la mauvaise extension
        old_path = os.path.join(labels_dir, filename)
        new_filename = filename.replace(".jpg.txt", ".txt")
        new_path = os.path.join(labels_dir, new_filename)
        
        os.rename(old_path, new_path)  # Renomme le fichier
        print(f"Renommé : {old_path} → {new_path}")

print("Renommage terminé ✅")

Renommé : C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/train\00000000.jpg.txt → C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/train\00000000.txt
Renommé : C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/train\00000001.jpg.txt → C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/train\00000001.txt
Renommé : C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/train\00000002.jpg.txt → C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/train\00000002.txt
Renommé : C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/train\00000003.jpg.txt → C:/Users/fanch/OneD

In [14]:
# Dossier contenant les annotations VAL
labels_dir = "C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/val"  # Modifie avec le bon chemin

# Parcours tous les fichiers du dossier
for filename in os.listdir(labels_dir):
    if filename.endswith(".jpg.txt"):  # Vérifie si le fichier a la mauvaise extension
        old_path = os.path.join(labels_dir, filename)
        new_filename = filename.replace(".jpg.txt", ".txt")
        new_path = os.path.join(labels_dir, new_filename)
        
        os.rename(old_path, new_path)  # Renomme le fichier
        print(f"Renommé : {old_path} → {new_path}")

print("Renommage terminé ✅")

Renommé : C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/val\00000048.jpg.txt → C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/val\00000048.txt
Renommé : C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/val\00000263.jpg.txt → C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/val\00000263.txt
Renommé : C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/val\00001977.jpg.txt → C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/val\00001977.txt
Renommé : C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/labels/val\00002106.jpg.txt → C:/Users/fanch/OneDrive/Bureau/Da

In [7]:
from ultralytics import YOLO
import numpy as np
from PIL import Image
import requests
from io import BytesIO
import cv2
import zipfile
import tarfile
import os
import pandas as pd
import json
import yaml

In [8]:
model = YOLO("yolov8n.pt")

100%|██████████| 6.25M/6.25M [00:00<00:00, 14.2MB/s]


In [ ]:
# Charger le modèle YOLOv8 pré-entraîné
model = YOLO("yolov8n.pt")

# Entraîner sur notre dataset
model.train(data="C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/yolo.yaml", epochs=5, imgsz=640, batch=16, freeze=10)

New https://pypi.org/project/ultralytics/8.3.100 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.99  Python-3.12.7 torch-2.6.0+cpu CPU (AMD Ryzen 5 5625U with Radeon Graphics)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=C:/Users/fanch/OneDrive/Bureau/Data_science/01_Jedha/Projets/Projet_final/dataset_jej_unzip/model/calcul/yolo.yaml, epochs=5, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train4, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augm

train: Scanning C:\Users\fanch\OneDrive\Bureau\Data_science\01_Jedha\Projets\Projet_final\dataset_jej_unzip\model\calcul\labels\train... 4983 images, 194 backgrounds, 0 corrupt: 100%|██████████| 4983/4983 [00:09<00:00, 501.01it/s]


train: New cache created: C:\Users\fanch\OneDrive\Bureau\Data_science\01_Jedha\Projets\Projet_final\dataset_jej_unzip\model\calcul\labels\train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Scanning C:\Users\fanch\OneDrive\Bureau\Data_science\01_Jedha\Projets\Projet_final\dataset_jej_unzip\model\calcul\labels\val... 1189 images, 1 backgrounds, 0 corrupt: 100%|██████████| 1189/1189 [00:02<00:00, 451.12it/s]


val: New cache created: C:\Users\fanch\OneDrive\Bureau\Data_science\01_Jedha\Projets\Projet_final\dataset_jej_unzip\model\calcul\labels\val.cache
Plotting labels to runs\detect\train4\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=9.3e-05, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
TensorBoard: model graph visualization added 
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs\detect\train4
Starting training for 5 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


        1/5         0G      2.734      5.528       2.59        155        640:  64%|██████▍   | 201/312 [26:13<14:28,  7.83s/it]


KeyboardInterrupt: 